# Report Label Extraction — RSNA Knee

Mines the twelve findings from the 4,349 training studies that have a radiology
report but no radiologist labels. Those mined labels are the training signal for
the imaging model; only 58 studies carry real labels, and those are reserved for
validation.

**Runtime:** A100 recommended. Roughly 45–75 min for the full corpus at 3
self-consistency samples per report.

**Data handling:** the model runs locally on this Colab GPU. Report text is
never sent to a hosted API.

### What you do

1. `Runtime → Change runtime type → A100 GPU`
2. Run every cell top to bottom.
3. Cell 6 is a checkpoint: it scores the extractor on the 58 gold studies and
   prints a reliability table. **Read it before running the full corpus** — if
   macro AUC is not comfortably above 0.70, stop and report the numbers back
   rather than spending an hour.
4. The last cell downloads `llm_labels.parquet`. Drop that into `data/meta/`.

## 1. Check the GPU and pick a model size

In [ ]:
import subprocess, torch

# nvidia-smi is absent on a CPU runtime, which is exactly the case the
# message below is for -- so this must not raise first.
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("nvidia-smi not found")

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100 GPU.")

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {VRAM_GB:.0f} GB")

# 14B needs ~28 GB in bf16; anything smaller gets the 7B.
if VRAM_GB >= 38:
    MODEL = "Qwen/Qwen2.5-14B-Instruct"
elif VRAM_GB >= 20:
    MODEL = "Qwen/Qwen2.5-7B-Instruct"
else:
    MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

print("model:", MODEL)

## 2. Install vLLM

Batched generation over ~13,000 prompts (4,349 reports × 3 samples). vLLM's
continuous batching is roughly an order of magnitude faster than a plain
`transformers` loop here.

Colab usually prompts to restart the runtime after this. **Restart, then skip
this cell and continue from cell 3.**

In [ ]:
!pip install -q vllm
print("installed -- restart the runtime if prompted, then continue from cell 3")

## 3. Load `train.csv`

Either mount Drive (if you put the file there) or upload it directly. It is
~5.7 MB.

In [ ]:
import os
import pandas as pd

TRAIN_CSV = None

# Try Drive first, then fall back to an upload prompt.
for candidate in ["/content/drive/MyDrive/knee/train.csv", "/content/train.csv"]:
    if os.path.exists(candidate):
        TRAIN_CSV = candidate
        break

if TRAIN_CSV is None:
    from google.colab import files
    print("Select train.csv ...")
    uploaded = files.upload()
    TRAIN_CSV = list(uploaded.keys())[0]

train = pd.read_csv(TRAIN_CSV)
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

gold = train[train["ACL"].notna()].reset_index(drop=True)
unlabelled = train[train["ACL"].isna()].reset_index(drop=True)
print(f"{len(train)} studies: {len(gold)} labelled, {len(unlabelled)} report-only")
assert len(gold) == 58, f"expected 58 gold studies, found {len(gold)}"

## 4. Extraction logic

Generated from `src/labels/llm_extract.py` in the repo — **do not edit here**,
edit the module and regenerate, or the notebook and the repo will disagree
about how responses are parsed.

In [ ]:
"""Prompt construction and response parsing for the LLM report extractor.

This module is deliberately free of GPU and model dependencies so it can be
unit-tested on a laptop. The notebook that runs the model embeds this file's
source directly, so the two cannot drift apart.

Contract matches src/labels/lexicon.py exactly -- `present` is 1, 0, or None,
and `severity` is a float or None -- so both extractors can be scored by the
same calibration code and fused without translation.
"""

import json
import re
from typing import Dict, List, Optional

# Findings, with the definition the model is asked to apply. Ground truth was
# adjudicated with explicit severity thresholds, so the definitions state what
# counts rather than leaving the model to guess a threshold.
FINDING_DEFINITIONS = {
    "ACL": "Anterior cruciate ligament tear, rupture, or significant injury. "
           "Not intact/normal ACL, not a mention of the PCL.",
    "MCL": "Medial collateral ligament tear, rupture, or sprain.",
    "Medial Meniscus": "Tear of the medial (internal) meniscus. Degeneration "
                       "without a tear does not count.",
    "Lateral Meniscus": "Tear of the lateral (external) meniscus. Degeneration "
                        "without a tear does not count.",
    "Medial OA": "Osteoarthritis, cartilage loss, or chondral thinning in the "
                 "MEDIAL femorotibial compartment.",
    "Lateral OA": "Osteoarthritis, cartilage loss, or chondral thinning in the "
                  "LATERAL femorotibial compartment.",
    "PF OA": "Osteoarthritis, chondromalacia, or cartilage loss in the "
             "PATELLOFEMORAL compartment (patella or trochlea).",
    "Effusion": "Joint effusion or increased intra-articular fluid.",
    "Synovitis": "Synovitis or synovial thickening/enhancement.",
    "Baker's": "Baker's cyst or popliteal cyst.",
    "Contusion": "Bone contusion or bone marrow oedema (excluding oedema that "
                 "is purely degenerative subchondral change).",
    "Fracture": "Any fracture, including stress, avulsion, and insufficiency "
                "fractures.",
}

PRESENCE_VALUES = {"yes": 1, "no": 0, "unclear": None}

# Severity and certainty map onto the same 0-1 scale the lexicon produces, so
# fusion compares like with like.
SEVERITY_SCORES = {"severe": 1.0, "moderate": 0.85, "minimal": 0.35, "none": 0.0}
CERTAINTY_FACTORS = {"definite": 1.0, "probable": 0.7, "possible": 0.5}

SYSTEM_PROMPT = (
    "You are a musculoskeletal radiologist extracting structured findings from "
    "knee MRI reports. Reports may be in any language, including English, "
    "Spanish, Turkish, German, Greek, Bulgarian, Croatian, Dutch, French, or "
    "Italian. Read the report in its original language. Do not translate it. "
    "Report only what the text states. Never infer a finding that the report "
    "does not describe."
)


def build_prompt(report: str) -> str:
    """The user turn: the report plus the schema the model must fill in."""
    definitions = "\n".join(
        f'  "{name}": {definition}' for name, definition in FINDING_DEFINITIONS.items()
    )

    return f"""Extract these twelve findings from the knee MRI report below.

FINDINGS:
{definitions}

For each finding return:
  "present": "yes" if the report describes it as present,
             "no" if the report explicitly states it is absent, normal, or intact,
             "unclear" if the report does not mention it at all.
  "severity": "severe", "moderate", "minimal", or "none".
             Use "minimal" for trace, mild, or slight findings.
  "certainty": "definite", "probable", or "possible".
             Use "possible" when the report hedges (suspicious for, cannot exclude).
  "evidence": the exact phrase from the report, in its original language, that
             supports your answer. Use "" when present is "unclear".

Critical: "unclear" and "no" are different. Use "no" ONLY when the report
positively asserts the structure is normal or the finding is absent. If the
report is simply silent about a finding, use "unclear".

REPORT:
\"\"\"
{report}
\"\"\"

Respond with ONLY a JSON object, no commentary, in exactly this form:
{{"ACL": {{"present": "...", "severity": "...", "certainty": "...", "evidence": "..."}}, ...}}
Include all twelve findings."""


def _empty_result() -> Dict[str, Dict]:
    return {
        name: {"present": None, "severity": None, "evidence": None}
        for name in FINDING_DEFINITIONS
    }


def _extract_json_block(text: str) -> Optional[str]:
    """Find the outermost JSON object in a response.

    Models wrap JSON in prose or fenced blocks often enough that locating the
    braces is more reliable than requiring a clean response.
    """
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start:index + 1]

    return None


def _score(severity: str, certainty: str) -> float:
    base = SEVERITY_SCORES.get(str(severity).strip().lower(), 0.85)
    factor = CERTAINTY_FACTORS.get(str(certainty).strip().lower(), 1.0)
    return base * factor


def parse_response(text: str) -> Dict[str, Dict]:
    """Turn one model response into the extractor contract.

    Unparseable responses and missing findings resolve to None rather than a
    guess -- a malformed generation must not be read as a confident negative.
    """
    result = _empty_result()
    if not isinstance(text, str) or not text.strip():
        return result

    block = _extract_json_block(text)
    if block is None:
        return result

    try:
        payload = json.loads(block)
    except json.JSONDecodeError:
        # Trailing commas are the usual culprit and are cheap to repair.
        repaired = re.sub(r",\s*([}\]])", r"\1", block)
        try:
            payload = json.loads(repaired)
        except json.JSONDecodeError:
            return result

    if not isinstance(payload, dict):
        return result

    for name in FINDING_DEFINITIONS:
        entry = payload.get(name)
        if not isinstance(entry, dict):
            continue

        raw_present = str(entry.get("present", "")).strip().lower()
        if raw_present not in PRESENCE_VALUES:
            continue

        present = PRESENCE_VALUES[raw_present]
        if present is None:
            result[name] = {"present": None, "severity": None, "evidence": None}
            continue

        severity = (
            _score(entry.get("severity", "moderate"), entry.get("certainty", "definite"))
            if present == 1 else 0.0
        )
        evidence = entry.get("evidence") or None
        result[name] = {
            "present": present,
            "severity": severity,
            "evidence": str(evidence)[:300] if evidence else None,
        }

    return result


def aggregate_samples(samples: List[Dict[str, Dict]]) -> Dict[str, Dict]:
    """Combine self-consistency samples into one result per finding.

    Majority vote decides presence; the vote fraction becomes the confidence
    that fusion uses to weight the sample. Findings where the model disagrees
    with itself are exactly the ones that should carry less weight.
    """
    if not samples:
        return {
            name: {"present": None, "severity": None, "evidence": None,
                   "confidence": 0.0}
            for name in FINDING_DEFINITIONS
        }

    aggregated = {}
    for name in FINDING_DEFINITIONS:
        votes = [s[name]["present"] for s in samples if name in s]
        counts = {
            1: votes.count(1),
            0: votes.count(0),
            None: votes.count(None),
        }
        winner = max(counts, key=lambda key: counts[key])
        confidence = counts[winner] / len(votes) if votes else 0.0

        matching = [
            s[name] for s in samples
            if name in s and s[name]["present"] == winner
        ]
        severities = [m["severity"] for m in matching if m["severity"] is not None]
        evidences = [m["evidence"] for m in matching if m["evidence"]]

        aggregated[name] = {
            "present": winner,
            "severity": (sum(severities) / len(severities)) if severities else None,
            "evidence": evidences[0] if evidences else None,
            "confidence": confidence,
        }

    return aggregated


def extract_finding_from_record(record: Dict, finding: str) -> Dict:
    """Adapter so calibrate.score_extractor can score cached LLM output."""
    entry = record.get(finding, {})
    return {
        "present": entry.get("present"),
        "severity": entry.get("severity"),
        "span": entry.get("evidence"),
    }

## 5. Load the model

`n=3` in the sampling parameters gives three independent samples per report in a
single pass. Majority vote decides each finding and the vote fraction becomes
the confidence weight used later during fusion — findings the model disagrees
with itself about should carry less weight.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MAX_REPORT_CHARS = 6000   # longest report in the corpus is ~4,750
N_SAMPLES = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(model=MODEL, dtype="bfloat16", gpu_memory_utilization=0.90,
          max_model_len=6144, trust_remote_code=True)

sampling = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=1400,
                          n=N_SAMPLES)


def to_chat(report):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(str(report)[:MAX_REPORT_CHARS])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True)


def run_extraction(reports, desc=""):
    prompts = [to_chat(r) for r in reports]
    print(f"generating {len(prompts)} x {N_SAMPLES} samples {desc}...")
    outputs = llm.generate(prompts, sampling)

    results = []
    for output in outputs:
        samples = [parse_response(o.text) for o in output.outputs]
        results.append(aggregate_samples(samples))
    return results

## 6. Checkpoint — score against the 58 gold studies

**Stop here and read the table.** This is the only honest measurement available.
The rule-based extractor already in the repo scores macro AUC 0.735, and its
weakest findings are the ones this model needs to carry:

| finding | lexicon AUC |
|---|---|
| Synovitis | 0.619 |
| PF OA | 0.646 |
| Contusion | 0.662 |
| Medial Meniscus | 0.678 |
| Lateral OA | 0.680 |

If the model lands below the lexicon on most findings, something is wrong with
the prompt or the parsing — report the table back rather than continuing.

In [ ]:
import numpy as np

gold_results = run_extraction(gold["Report"].tolist(), "(gold 58)")


def auc(y_true, scores):
    y_true = np.asarray(y_true); scores = np.asarray(scores, dtype=float)
    pos = y_true == 1
    n_pos, n_neg = int(pos.sum()), int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1)
    unique, inverse, counts = np.unique(scores, return_inverse=True,
                                        return_counts=True)
    sums = np.zeros(len(unique)); np.add.at(sums, inverse, ranks)
    ranks = (sums / counts)[inverse]
    return (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


print(f"\n{'finding':18s} {'n+':>3s} {'ment':>5s} {'prec':>5s} {'rec':>5s} {'AUC':>6s}")
print("-" * 48)
aucs = []
for name in LABELS:
    truth = gold[name].astype(int).to_numpy()
    present = np.array([r[name]["present"] if r[name]["present"] is not None else -1
                        for r in gold_results])
    severity = np.array([r[name]["severity"] if r[name]["severity"] is not None else 0.0
                         for r in gold_results])
    pred = (present == 1).astype(int)
    tp = int(((pred == 1) & (truth == 1)).sum())
    fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    a = auc(truth, severity); aucs.append(a)
    print(f"{name:18s} {truth.sum():3d} {(present != -1).mean():5.0%} "
          f"{prec:5.2f} {rec:5.2f} {a:6.3f}")
print("-" * 48)
print(f"macro AUC: {np.nanmean(aucs):.3f}   (lexicon baseline: 0.735)")

## 7. Full corpus

Only run this once the table above looks right. This is the long cell.

In [ ]:
import time

start = time.time()
mined = run_extraction(unlabelled["Report"].tolist(), "(4,349 reports)")
print(f"done in {(time.time() - start) / 60:.1f} min")

## 8. Save and download

One row per study: `<finding>` (1/0/NaN), `<finding>_severity`,
`<finding>_confidence`, and the evidence span. The evidence is what makes a
disagreement between the two extractors auditable instead of mysterious, so it
is kept rather than discarded.

In [ ]:
def to_frame(results, ids):
    rows = []
    for result in results:
        row = {}
        for name in LABELS:
            entry = result[name]
            row[name] = entry["present"]
            row[f"{name}_severity"] = entry["severity"]
            row[f"{name}_confidence"] = entry.get("confidence")
            row[f"{name}_evidence"] = entry.get("evidence")
        rows.append(row)
    frame = pd.DataFrame(rows)
    frame.insert(0, "StudyInstanceUID", list(ids))
    return frame


mined_frame = to_frame(mined, unlabelled["StudyInstanceUID"])
gold_frame = to_frame(gold_results, gold["StudyInstanceUID"])

mined_frame.to_parquet("llm_labels.parquet", index=False)
gold_frame.to_parquet("llm_labels_gold.parquet", index=False)

print(mined_frame.shape)
print("\npositive rate per finding (mined):")
for name in LABELS:
    values = mined_frame[name]
    print(f"  {name:18s} pos {(values == 1).mean():5.1%}  "
          f"neg {(values == 0).mean():5.1%}  unclear {values.isna().mean():5.1%}")

from google.colab import files
files.download("llm_labels.parquet")
files.download("llm_labels_gold.parquet")

## Next

Both parquet files go into `data/meta/`. Fusion then combines these with the
rule-based extractor: agreement becomes a hard target at full weight,
disagreement becomes a soft target at reduced weight, and silence is filled with
the per-finding `P(positive | unmentioned)` prior measured on the gold studies —
which ranges from 0.05 for MCL to 0.58 for Effusion, and is the reason silence
cannot simply be treated as a negative.